# AR × mutual_info — Test regression applicability domain

This notebook replays the frozen HStack1024 + SelectKBest regression variant, builds the applicability-domain reference from Train+Validation, and evaluates the locked Test split. It creates one summary CSV for each configured threshold multiplier. No model, scaler, or selector is fitted.


## Method

Run target: **AR × mutual_info**

1. Recreate and verify the locked 60/20/20 split.
2. Extract fresh SMILES, SELFIES, Graph, and Fingerprint features.
3. Apply the frozen MinMaxScaler and frozen SelectKBest selector.
4. Combine Train+Validation selected features as the AD reference.
5. Predict the locked Test split with the frozen final regressor.
6. For AD k=3…25, classify Test rows as IND/OOD with Euclidean kNN.
7. Report R2, RMSE, MAE, ME, Pearson, and Spearman on IND rows.
8. Write one CSV per threshold multiplier.

The `No AD` row uses every Test row. `ME` is prediction minus observation.


In [ ]:
# Configuration: edit AD_THRESHOLD_MULTIPLIERS here.
RUN_DATASET = 'AR'
RUN_METHOD = 'mutual_info'
TOLERANCE = 0.001

# One CSV is created for each multiplier. Examples:
#   (0.5, 2.0) -> two CSV files
#   (1.0,)     -> one CSV file
AD_THRESHOLD_MULTIPLIERS = (0.5, 2.0)
AD_NEIGHBOR_K_VALUES = tuple(range(3, 26))

GITHUB_REPO_URL = "https://github.com/bigmanaschai/RegressionPipeline_hstack1024_FS.git"
GITHUB_REF = "main"    # branch, tag, or commit available to git clone
PROJECT_ROOT_OVERRIDE = ""  # existing checkout; takes priority over clone
BASE_ROOT_OVERRIDE = ""     # blank = standard Kaggle base-library path
FS_ROOT_OVERRIDE = ""       # blank = materials in the GitHub checkout
SPLIT_ROOT = "/kaggle/input"
OUTPUT_ROOT_OVERRIDE = ""

assert RUN_DATASET in {"AR", "ER", "GR", "PR"}
assert RUN_METHOD in {"mutual_info", "pearson"}
assert AD_THRESHOLD_MULTIPLIERS
assert all(float(value) >= 0.0 for value in AD_THRESHOLD_MULTIPLIERS)
assert tuple(AD_NEIGHBOR_K_VALUES) == tuple(range(3, 26))


In [ ]:
# Resolve or clone the GitHub project without modifying Kaggle Inputs.
from pathlib import Path
import subprocess

EXPECTED_RELATIVE = Path("standard_pipeline/RegressionPipeline_hstack1024_FS")

def find_extension_root(project_root):
    """Support both the monorepo layout and this package's standalone GitHub repo."""
    nested = project_root / EXPECTED_RELATIVE
    if nested.is_dir():
        return nested
    if (
        (project_root / "src" / "hstack1024_fs_pipeline").is_dir()
        and (project_root / "materials" / "hstack1024-fs-extension").is_dir()
        and (project_root / "pyproject.toml").is_file()
    ):
        return project_root
    raise FileNotFoundError(
        f"RegressionPipeline_hstack1024_FS source not found under: {project_root}"
    )

if PROJECT_ROOT_OVERRIDE:
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
elif (Path.cwd() / EXPECTED_RELATIVE).is_dir():
    PROJECT_ROOT = Path.cwd().resolve()
else:
    if not GITHUB_REPO_URL:
        raise ValueError(
            "Set GITHUB_REPO_URL to the repository pushed from this materials folder, "
            "or set PROJECT_ROOT_OVERRIDE to an existing checkout."
        )
    checkout = Path("/kaggle/working/hstack1024-fs-repository")
    if not checkout.is_dir():
        subprocess.check_call(
            ["git", "clone", "--no-checkout", "--filter=blob:none", GITHUB_REPO_URL, str(checkout)]
        )
    subprocess.check_call(
        ["git", "-C", str(checkout), "fetch", "--depth", "1", "origin", GITHUB_REF]
    )
    subprocess.check_call(
        ["git", "-C", str(checkout), "checkout", "--detach", "FETCH_HEAD"]
    )
    PROJECT_ROOT = checkout.resolve()

EXTENSION_ROOT = find_extension_root(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)
print("Extension   :", EXTENSION_ROOT)


In [ ]:
# Install only missing optional deep-inference dependencies.
import importlib.util
import os
import subprocess
import sys

# These frozen inference pipelines do not use experiment tracking.
os.environ.setdefault("WANDB_MODE", "disabled")
os.environ.setdefault("WANDB_SILENT", "true")

required_modules = {
    "transformers": "transformers==4.48.3",
    "sentencepiece": "sentencepiece",
    "selfies": "selfies",
    "rdkit": "rdkit",
    "deepchem": "deepchem",
    "torch_geometric": "torch-geometric",
    "skfp": "git+https://github.com/plenoi/scikit-finger-plenoi.git@master",
}
missing = [package for module, package in required_modules.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

from transformers import AutoModel, AutoTokenizer

def pin_remote_code_revision(auto_class):
    original = auto_class.from_pretrained
    if getattr(original, "_hstack_revision_pinned", False):
        return
    def pinned(pretrained_model_name_or_path, *args, **kwargs):
        revision = kwargs.get("revision")
        if revision:
            kwargs.setdefault("code_revision", revision)
        return original(pretrained_model_name_or_path, *args, **kwargs)
    pinned._hstack_revision_pinned = True
    auto_class.from_pretrained = pinned

pin_remote_code_revision(AutoTokenizer)
pin_remote_code_revision(AutoModel)
print("Optional dependencies: OK")


In [ ]:
# Resolve packages and verify the small GitHub material bundle byte-for-byte.
import hashlib
import importlib
import json
import sys

BASE_KAGGLE_ROOT = Path("/kaggle/input/datasets/manaschaiaonon/hstack1024-pipeline-libs")
LOCAL_BASE_ROOT = PROJECT_ROOT / "standard_pipeline/frozen_hstack1024_ridge/kaggle_dataset/hstack1024-pipeline-libs"
BASE_ROOT_CANDIDATE = Path(BASE_ROOT_OVERRIDE).expanduser() if BASE_ROOT_OVERRIDE else (
    BASE_KAGGLE_ROOT if BASE_KAGGLE_ROOT.is_dir() else LOCAL_BASE_ROOT
)

def find_base_artifact_root(candidate):
    """Accept direct or one-level/nested Kaggle uploads of the flat bundle."""
    candidate = Path(candidate)
    direct = candidate / "checkpoints" / "model_smiles_AR.pt"
    if direct.is_file():
        return candidate
    roots = sorted({
        path.parent.parent.resolve()
        for path in candidate.rglob("model_smiles_AR.pt")
        if path.parent.name == "checkpoints"
    })
    if len(roots) == 1:
        return roots[0]
    return candidate

BASE_ROOT = find_base_artifact_root(BASE_ROOT_CANDIDATE)
FS_ROOT = Path(FS_ROOT_OVERRIDE).expanduser() if FS_ROOT_OVERRIDE else (
    EXTENSION_ROOT / "materials/hstack1024-fs-extension"
)
BASE_CODE_SRC = FS_ROOT / "base_src"
FS_SRC = FS_ROOT / "src"
if not BASE_ROOT.is_dir():
    raise FileNotFoundError(f"Attach base Kaggle library; root not found: {BASE_ROOT}")
if not (BASE_CODE_SRC / "hstack1024_pipeline").is_dir():
    raise FileNotFoundError(f"Vendored base source package not found: {BASE_CODE_SRC}")
if not (FS_SRC / "hstack1024_fs_pipeline").is_dir():
    raise FileNotFoundError(f"FS source package not found: {FS_SRC}")

manifest = json.loads((FS_ROOT / "MANIFEST.json").read_text(encoding="utf-8"))
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()
for relative, metadata in manifest["files"].items():
    path = FS_ROOT / relative
    if not path.is_file() or sha256(path) != metadata["sha256"]:
        raise RuntimeError(f"FS material missing or stale: {relative}")

for module_name in list(sys.modules):
    if (
        module_name == "hstack1024_pipeline"
        or module_name.startswith("hstack1024_pipeline.")
        or module_name == "hstack1024_fs_pipeline"
        or module_name.startswith("hstack1024_fs_pipeline.")
    ):
        del sys.modules[module_name]
sys.path[:0] = [str(FS_SRC), str(BASE_CODE_SRC)]
importlib.invalidate_caches()

import hstack1024_pipeline
import hstack1024_fs_pipeline

expected_imports = {
    "hstack1024_pipeline": (hstack1024_pipeline, BASE_CODE_SRC / "hstack1024_pipeline"),
    "hstack1024_fs_pipeline": (hstack1024_fs_pipeline, FS_SRC / "hstack1024_fs_pipeline"),
}
for package_name, (package, expected_dir) in expected_imports.items():
    imported_file = getattr(package, "__file__", None)
    if imported_file is None or Path(imported_file).resolve().parent != expected_dir.resolve():
        raise RuntimeError(
            f"Imported {package_name} from {imported_file!r}; expected {expected_dir}"
        )

base_config = importlib.import_module("hstack1024_pipeline.config")
print("Base config    :", base_config.__file__)
print(
    "Artifact dirs :",
    {
        directory: (BASE_ROOT / directory).is_dir()
        for directory in ("checkpoints", "models", "transformers")
    },
)
print("Flat bundle    :", base_config.is_flat_kaggle_bundle(BASE_ROOT))

base_contract = getattr(hstack1024_pipeline, "PIPELINE_CONTRACT_VERSION", None)
if base_contract != "fresh-raw-split-extraction-cpu-v2":
    raise RuntimeError(
        "Incompatible base HStack1024 contract: "
        f"{base_contract!r} from {hstack1024_pipeline.__file__}"
    )
fs_contract = getattr(hstack1024_fs_pipeline, "PIPELINE_CONTRACT_VERSION", None)
if fs_contract != "fresh-raw-split-extraction-selectkbest-cpu-v1":
    raise RuntimeError(
        "Incompatible HStack1024 FS contract: "
        f"{fs_contract!r} from {hstack1024_fs_pipeline.__file__}"
    )
print("Base root     :", BASE_ROOT)
print("FS materials  :", FS_ROOT)
print("Inference device: CPU")


In [ ]:
# Load the manifest-verified AD runtime from the same FS source bundle.
ad_source = FS_SRC / "ad_test_regression" / "pipeline.py"
if not ad_source.is_file():
    raise RuntimeError(
        "Test-regression AD runtime is absent from the material bundle: "
        f"{ad_source}. Rebuild materials, push them, restart the session, and Run All."
    )

for module_name in list(sys.modules):
    if module_name == "ad_test_regression" or module_name.startswith("ad_test_regression."):
        del sys.modules[module_name]
importlib.invalidate_caches()

import ad_test_regression
imported_ad = Path(ad_test_regression.__file__).resolve().parent
expected_ad = (FS_SRC / "ad_test_regression").resolve()
if imported_ad != expected_ad:
    raise RuntimeError(f"Imported AD runtime from {imported_ad}; expected {expected_ad}")
print("Test AD runtime:", imported_ad)


In [ ]:
# Preflight raw, split-oracle, deep-model, final-model, and reference inputs.
from hstack1024_pipeline.config import DATASETS, FAMILY_ORDER, checkpoint_path, fingerprint_transformer_path, raw_csv_path
from hstack1024_fs_pipeline.cli import discover_split_paths
from hstack1024_fs_pipeline.config import FS_METHODS, bundle_path, get_variant, reference_path

datasets = tuple(DATASETS) if RUN_DATASET == "ALL" else (RUN_DATASET,)
methods = FS_METHODS if RUN_METHOD == "ALL" else (RUN_METHOD,)
split_root = Path(SPLIT_ROOT)
split_paths_by_dataset = {dataset: discover_split_paths(dataset, split_root) for dataset in datasets}
raw_csv_paths = {dataset: raw_csv_path(dataset, BASE_ROOT) for dataset in datasets}

for dataset in datasets:
    if not raw_csv_paths[dataset].is_file():
        raise FileNotFoundError(raw_csv_paths[dataset])
    for family in FAMILY_ORDER:
        if not checkpoint_path(dataset, family, BASE_ROOT).is_file():
            raise FileNotFoundError(checkpoint_path(dataset, family, BASE_ROOT))
    if not fingerprint_transformer_path(dataset, BASE_ROOT).is_file():
        raise FileNotFoundError(fingerprint_transformer_path(dataset, BASE_ROOT))
    for method in methods:
        spec = get_variant(dataset, method)
        if not bundle_path(dataset, method, fs_root=FS_ROOT).is_file():
            raise FileNotFoundError(bundle_path(dataset, method, fs_root=FS_ROOT))
        if not reference_path(dataset, method, fs_root=FS_ROOT).is_file():
            raise FileNotFoundError(reference_path(dataset, method, fs_root=FS_ROOT))
        print(
            "Selected variant:", spec.variant_id,
            f"Model={spec.model_class}", f"FS_k={spec.selected_k}",
            f"Reference_Test_R2={spec.reference_test_r2:.12f}",
            f"Sheet={spec.reference_sheet}", f"Rule={spec.selection_rule}",
        )
print("Preflight: OK", datasets, methods)


In [ ]:
# Run Test-set regression AD and write one CSV per threshold multiplier.
import pandas as pd
from ad_test_regression.pipeline import run_test_ad_variant

target_name = f"{RUN_DATASET}_{RUN_METHOD}"
default_output = Path("/kaggle/working") / f"AD_Test_Regression_{target_name}"
OUTPUT_ROOT = Path(OUTPUT_ROOT_OVERRIDE).expanduser() if OUTPUT_ROOT_OVERRIDE else default_output

output_paths = run_test_ad_variant(
    RUN_DATASET,
    RUN_METHOD,
    OUTPUT_ROOT,
    base_root=BASE_ROOT,
    fs_root=FS_ROOT,
    repo_root=PROJECT_ROOT,
    raw_csv=raw_csv_paths[RUN_DATASET],
    split_paths=split_paths_by_dataset[RUN_DATASET],
    threshold_multipliers=AD_THRESHOLD_MULTIPLIERS,
    k_values=AD_NEIGHBOR_K_VALUES,
    tolerance=TOLERANCE,
)

print("Output root:", OUTPUT_ROOT)
for multiplier, output_path in output_paths.items():
    frame = pd.read_csv(output_path)
    print(f"threshold multiplier={multiplier}: {output_path}")
    display(frame)


## Output interpretation

Each CSV contains `No AD` followed by exactly one row for every AD k from 3 through 25. `Coverage = INDs / N_Test` and `OODs = N_Test - INDs`. A larger threshold multiplier must not reduce Coverage for the same AD k.
